# Funding Calls Ingestion

This notebook builds the retrieval knowledge base from raw funding call PDFs. The blocks below load source documents, clean and segment text, generate embeddings, and persist the final corpus in ChromaDB. Its role in the project is to create the indexed data layer that all downstream retrieval, ranking, and explanation steps depend on.

## Imports

In [2]:
import hashlib
import re
from pathlib import Path
from typing import Dict, List

import chromadb
from chromadb.utils import embedding_functions
from pypdf import PdfReader
from tqdm.auto import tqdm

/home/lburmazevic/Projects/EY-AI-Techniques-Solution/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Centralizing ingestion settings. It is the control panel for reproducibility: changing values here determines where data is read/written and how document granularity is prepared for retrieval quality.

In [3]:
PROJECT_ROOT = Path.cwd()
PDF_DIR = PROJECT_ROOT / "docs" / "fundingcalls"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "funding_calls"

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 250
MIN_CHUNK_CHARS = 120

RESET_COLLECTION = True
EXCLUDE_FILES = set()

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF directory: {PDF_DIR}")
print(f"Chroma directory: {CHROMA_DIR}")

Project root: /home/lburmazevic/Projects/EY-AI-Techniques-Solution
PDF directory: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/fundingcalls
Chroma directory: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/data/chroma


## Source Files

List all PDF inputs selected for ingestion and validates that source files exist before processing starts.

In [4]:
pdf_paths = sorted([p for p in PDF_DIR.glob("*.pdf") if p.name not in EXCLUDE_FILES])

print(f"Found {len(pdf_paths)} PDF files")
for path in pdf_paths:
    print(" -", path.name)

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {PDF_DIR}")

Found 60 PDF files
 - Centri Nazionali Tematiche PNRR.pdf
 - Ecosistemi dell'Innovazione PNRR.pdf
 - HORIZON-WIDERA-2021-ERA-01-09.pdf
 - HORIZON-WIDERA-2021-ERA-01-20.pdf
 - HORIZON-WIDERA-2021-ERA-01-30.pdf
 - HORIZON-WIDERA-2021-ERA-01-32.pdf
 - HORIZON-WIDERA-2021-ERA-01-33.pdf
 - HORIZON-WIDERA-2021-ERA-01-40.pdf
 - HORIZON-WIDERA-2021-ERA-01-41.pdf
 - HORIZON-WIDERA-2021-ERA-01-43.pdf
 - HORIZON-WIDERA-2021-ERA-01-44.pdf
 - HORIZON-WIDERA-2021-ERA-01-45.pdf
 - HORIZON-WIDERA-2021-ERA-01-50.pdf
 - HORIZON-WIDERA-2021-ERA-01-60.pdf
 - HORIZON-WIDERA-2021-ERA-01-61.pdf
 - HORIZON-WIDERA-2021-ERA-01-70.pdf
 - HORIZON-WIDERA-2021-ERA-01-80.pdf
 - HORIZON-WIDERA-2021-ERA-01-81.pdf
 - HORIZON-WIDERA-2021-ERA-01-90.pdf
 - HORIZON-WIDERA-2021-ERA-01-91.pdf
 - HORIZON-WIDERA-2022-ERA-01-10.pdf
 - HORIZON-WIDERA-2022-ERA-01-30.pdf
 - HORIZON-WIDERA-2022-ERA-01-31.pdf
 - HORIZON-WIDERA-2022-ERA-01-32.pdf
 - HORIZON-WIDERA-2022-ERA-01-40.pdf
 - HORIZON-WIDERA-2022-ERA-01-41.pdf
 - HORIZON-WID

## Text Extraction

Defining and executes page-level extraction and normalization from funding call PDFs. It produces a clean intermediate representation (non-empty pages with metadata), which is the foundation for stable chunking and high-quality embeddings.

In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_pages_from_pdf(pdf_path: Path) -> List[Dict]:
    rows: List[Dict] = []
    reader = PdfReader(str(pdf_path))

    for page_idx, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        text = clean_text(raw_text)

        if len(text) < 40:
            continue

        rows.append(
            {
                "source_file": pdf_path.name,
                "source_path": str(pdf_path),
                "page": page_idx,
                "text": text,
            }
        )

    return rows

In [ ]:
page_rows: List[Dict] = []
for pdf_path in tqdm(pdf_paths, desc="Extracting PDF pages"):
    page_rows.extend(extract_pages_from_pdf(pdf_path))

print(f"Extracted {len(page_rows)} non-empty pages")
page_rows[0] if page_rows else "No text extracted"  

Extracting PDF pages:   2%|▏         | 1/60 [00:00<00:06,  9.59it/s]

Extracting PDF pages: 100%|██████████| 60/60 [00:11<00:00,  5.04it/s]

Extracted 240 non-empty pages


{'source_file': 'Centri Nazionali Tematiche PNRR.pdf',
 'source_path': '/home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/fundingcalls/Centri Nazionali Tematiche PNRR.pdf',
 'page': 1,
 'text': '1 Avviso pubblico per la presentazione di Proposte di intervento per il Potenziamento di strutture di ricerca e creazione di “campioni nazionali” di R&S su alcune Key Enabling Technologies da finanziare nell’ambito del Piano Nazionale di Ripresa e Resilienza, Missione 4 Componente 2 Investimento 1.4 finanziato dall’Unione europea – NextGenerationEU. Allegato A - Tematiche (articolo 1 comma 1 dell’Avviso)'}

## Chunking

This block transforms cleaned page text into overlapping chunks and prepares document IDs and metadata records for indexing. Its role is to optimize retrieval recall/precision by balancing chunk size, overlap, and minimum-content thresholds.

In [7]:
def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if overlap >= size:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")

    chunks: List[str] = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + size, text_len)
        chunk = text[start:end].strip()

        if len(chunk) >= MIN_CHUNK_CHARS:
            chunks.append(chunk)

        if end == text_len:
            break

        start = end - overlap

    return chunks

In [8]:
documents: List[str] = []
metadatas: List[Dict] = []
ids: List[str] = []

for row in tqdm(page_rows, desc="Chunking pages"):
    chunks = chunk_text(row["text"])

    for chunk_idx, chunk in enumerate(chunks):
        raw_id = f"{row['source_file']}|{row['page']}|{chunk_idx}|{len(chunk)}"
        chunk_id = hashlib.md5(raw_id.encode("utf-8")).hexdigest()

        documents.append(chunk)
        metadatas.append(
            {
                "source_file": row["source_file"],
                "source_path": row["source_path"],
                "page": row["page"],
                "chunk_index": chunk_idx,
                "chunk_chars": len(chunk),
            }
        )
        ids.append(chunk_id)

print(f"Created {len(documents)} chunks")
if documents:
    print("Sample chunk preview:")
    print(documents[0][:500])

Chunking pages: 100%|██████████| 240/240 [00:00<00:00, 96420.78it/s]

Created 509 chunks
Sample chunk preview:
1 Avviso pubblico per la presentazione di Proposte di intervento per il Potenziamento di strutture di ricerca e creazione di “campioni nazionali” di R&S su alcune Key Enabling Technologies da finanziare nell’ambito del Piano Nazionale di Ripresa e Resilienza, Missione 4 Componente 2 Investimento 1.4 finanziato dall’Unione europea – NextGenerationEU. Allegato A - Tematiche (articolo 1 comma 1 dell’Avviso)


## ChromaDB Ingestion

Instantiating the embedding function, connecting to persistent ChromaDB storage, and writing chunk vectors with metadata.

In [9]:
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="Qwen/Qwen3-Embedding-0.6B"
)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if RESET_COLLECTION:
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn,
    metadata={"description": "Funding call chunks for RAG retrieval"},
)

BATCH_SIZE = 128
for i in tqdm(range(0, len(documents), BATCH_SIZE), desc="Upserting to Chroma"):
    collection.upsert(
        ids=ids[i:i + BATCH_SIZE],
        documents=documents[i:i + BATCH_SIZE],
        metadatas=metadatas[i:i + BATCH_SIZE],
    )

print("Collection name:", COLLECTION_NAME)
print("Stored chunks:", collection.count())

Upserting to Chroma: 100%|██████████| 4/4 [1:35:24<00:00, 1431.14s/it]

Collection name: funding_calls
Stored chunks: 509


## Ingestion Validation

Verifying ingestion integrity by comparing expected chunk counts, stored counts, and ID uniqueness

In [10]:
expected_chunks = len(documents)
stored_chunks = collection.count()

print("Expected chunks:", expected_chunks)
print("Stored chunks:", stored_chunks)
print("Unique IDs:", len(set(ids)))
print("Count match:", expected_chunks == stored_chunks)

if expected_chunks != stored_chunks:
    raise ValueError("Chunk count mismatch: some chunks may not have been written to ChromaDB.")

Expected chunks: 509
Stored chunks: 509
Unique IDs: 509
Count match: True
